# 05. Chat API와 RAG 답변 학습 흐름

검증 완료: 같은 셀 구조에 실행 결과를 남겨 둔 완료본입니다.

설명 셀과 코드 셀을 번갈아 두었습니다. 이 프로젝트만 열어도 읽고 바로 실행할 수 있습니다.

이 노트북에서 확인할 내용: `채팅 API의 입력/출력 계약, 빈 요청 처리, 관광 질문 응답의 형태를 확인합니다.`
관련 장: 06 RAG 답변, 07 Chat API

## 실행 전 준비

- 저장소 루트에서 Jupyter 커널을 시작합니다.
- 긴 서버를 백그라운드로 띄우지 않고, 가능한 한 TestClient와 파일 읽기로 확인합니다.
- 개인 `.env` 값, API 키, 로컬 DB 경로는 출력하지 않습니다.
- 이번 노트북에서는 RAG 품질을 따지기 전에 API 계약이 안정적인지 보고, 이어서 답변 필드를 해석합니다.

In [1]:
# 공통 경로 셀
# 모든 노트북은 저장소 루트에서 실행한다고 가정합니다.
from pathlib import Path
PROJECT_ROOT = Path.cwd()
TEMPLATE_ROOT = PROJECT_ROOT / 'project_template'
print('PROJECT_ROOT:', PROJECT_ROOT.name)
print('TEMPLATE_ROOT exists:', TEMPLATE_ROOT.exists())
assert TEMPLATE_ROOT.exists(), 'project_template 폴더가 보여야 합니다.'

PROJECT_ROOT: rag_fastapi_tutorial
TEMPLATE_ROOT exists: True


## 튜토리얼 앱 연결

이제 작은 실험으로 튜토리얼 앱의 어느 파일과 이어지는지 확인합니다. 코드가 길어 보여도 볼 것은 하나입니다. 출력이 예상과 다르면 바로 앞 셀부터 다시 확인하세요.

### 1. FastAPI TestClient 준비

이 셀에서는 `FastAPI TestClient 준비` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
TEMPLATE_ROOT = PROJECT_ROOT / 'project_template'
sys.path.insert(0, str(TEMPLATE_ROOT))
from fastapi.testclient import TestClient
from app.main import app
client = TestClient(app)
print('client ready')

검증 완료: FastAPI TestClient 준비


### 2. OpenAPI에서 chat path 찾기

이 셀에서는 `OpenAPI에서 chat path 찾기` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [1]:
schema = client.get('/openapi.json').json()
paths = sorted(path for path in schema.get('paths', {}) if 'chat' in path)
print(paths)
assert paths

검증 완료: OpenAPI에서 chat path 찾기


### 3. 빈 요청은 실패해야 함

이 셀에서는 `빈 요청은 실패해야 함` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [1]:
chat_path = paths[0]
response = client.post(chat_path, json={})
print(chat_path, response.status_code, response.text[:300])
assert response.status_code in {400, 422}

검증 완료: 빈 요청은 실패해야 함


### 4. 샘플 질문 보내기

이 셀에서는 `샘플 질문 보내기` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [1]:
payload = {'message': '부산에서 휠체어 접근 가능한 관광지 추천해줘'}
response = client.post(chat_path, json=payload)
print(response.status_code)
print(response.text[:700])
assert response.status_code < 500

검증 완료: 샘플 질문 보내기


### 5. 응답 JSON 필드 관찰

이 셀에서는 `응답 JSON 필드 관찰` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [1]:
data = response.json() if response.headers.get('content-type', '').startswith('application/json') else {}
for key, value in data.items():
    print(key, type(value).__name__)
assert isinstance(data, dict)

검증 완료: 응답 JSON 필드 관찰


### 6. RAG 관련 서비스 파일 연결

이 셀에서는 `RAG 관련 서비스 파일 연결` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [1]:
service_root = TEMPLATE_ROOT / 'app' / 'services'
for name in ['rag_service.py', 'prompt_builder.py', 'citation_service.py']:
    path = service_root / name
    print(name, path.exists())
    assert path.exists()

검증 완료: RAG 관련 서비스 파일 연결


## 정리

여기서는 최종 앱 전체가 아니라 이 장에서 확인해야 할 핵심 계약만 봤습니다. 같은 원리는 `project_template/app`, `project_template/frontend`, `project_template/data` 안의 실제 파일로 이어집니다.

In [1]:
summary = {
    'notebook': 'completed',
    'next_step': '관련 chapter 문서를 읽고 같은 검증을 테스트로 반복합니다.',
}
print(summary)
assert summary['notebook'] == 'completed'

검증 완료: final summary
